# 01 — Data Collection

This notebook downloads lap-by-lap data for every race in the **2025 F1 season** (24 rounds)  
using the [FastF1](https://github.com/theOehrly/Fast-F1) library.

**Output:** `../data/raw/all_grand_prix_laps_2025.csv`

| Step | Description |
|------|-------------|
| 1 | Configure FastF1 cache |
| 2 | Load each race session and extract laps |
| 3 | Validate coverage and save |

> **Re-run safety:** The script detects which rounds are already saved and only fetches new ones.

In [ ]:
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

import fastf1
import pandas as pd

# Add src/ to path so we can import shared helpers
sys.path.insert(0, os.path.join('..', 'src'))
from data_processing import load_lap_data

print(f'FastF1 version: {fastf1.__version__}')
print(f'pandas version:  {pd.__version__}')

## 1 · Configure Cache

FastF1 caches API responses locally. This keeps subsequent runs fast and avoids hitting the API repeatedly.

In [ ]:
CACHE_DIR  = os.path.join('..', 'FastF1Cache')
OUTPUT_PATH = os.path.join('..', 'data', 'raw', 'all_grand_prix_laps_2025.csv')
YEAR = 2025

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

print(f'Cache:  {os.path.abspath(CACHE_DIR)}')
print(f'Output: {os.path.abspath(OUTPUT_PATH)}')

## 2 · Inspect the 2025 Season Calendar

FastF1 exposes the full event schedule so we know exactly which rounds to fetch.

In [ ]:
schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
race_events = schedule[schedule['EventFormat'].isin(['conventional', 'sprint'])].copy()
race_events = race_events[['RoundNumber', 'EventName', 'EventDate', 'EventFormat', 'Country']]
print(f'Total race events in {YEAR}: {len(race_events)}')
race_events

## 3 · Download All Race Laps

We iterate over every round, load the Race session, and keep the lap-level DataFrame.  
Rounds already present in the output file are automatically skipped.

In [ ]:
def get_existing_rounds(path: str) -> set:
    """Return round numbers already saved in the CSV."""
    if not os.path.exists(path):
        return set()
    existing = pd.read_csv(path, usecols=['RoundNumber'])
    return set(existing['RoundNumber'].dropna().astype(int).unique())


def fetch_race_laps(year: int, round_num: int) -> pd.DataFrame | None:
    """Load laps for a single race round."""
    try:
        session = fastf1.get_session(year, round_num, 'R')
        session.load(telemetry=False, weather=False, messages=False)
        laps = session.laps.copy()
        if laps.empty:
            return None
        laps['RoundNumber'] = round_num
        laps['EventName'] = session.event['EventName']
        return laps
    except Exception as e:
        print(f'  ERROR: {e}')
        return None


already_done = get_existing_rounds(OUTPUT_PATH)
all_rounds = sorted(race_events['RoundNumber'].astype(int).tolist())
to_fetch = [r for r in all_rounds if r not in already_done]

print(f'Rounds already saved : {sorted(already_done) if already_done else "none"}')
print(f'Rounds to fetch      : {to_fetch}')

In [ ]:
collected = []

for i, rnd in enumerate(to_fetch, 1):
    event_name = race_events.loc[race_events['RoundNumber'] == rnd, 'EventName'].values[0]
    print(f'[{i:02d}/{len(to_fetch):02d}] Round {rnd:02d} — {event_name} ...', end=' ', flush=True)
    t0 = time.time()
    df = fetch_race_laps(YEAR, rnd)
    if df is not None:
        collected.append(df)
        print(f'OK  ({len(df)} laps, {time.time()-t0:.1f}s)')
    else:
        print(f'SKIPPED ({time.time()-t0:.1f}s)')

print(f'\nFetched {len(collected)} new race sessions.')

## 4 · Merge & Save

In [ ]:
if collected:
    new_data = pd.concat(collected, ignore_index=True)

    if os.path.exists(OUTPUT_PATH):
        existing_df = pd.read_csv(OUTPUT_PATH, low_memory=False)
        final = pd.concat([existing_df, new_data], ignore_index=True)
    else:
        final = new_data

    final = final.sort_values(['RoundNumber', 'Driver', 'LapNumber'])
    final.to_csv(OUTPUT_PATH, index=False)
    print(f'Saved {len(final):,} total laps to {OUTPUT_PATH}')
else:
    print('No new data to save.')
    final = pd.read_csv(OUTPUT_PATH, low_memory=False)
    print(f'Loaded existing file: {len(final):,} laps')

## 5 · Validate Coverage

In [ ]:
coverage = (
    final.groupby(['RoundNumber', 'EventName'])
    .agg(
        drivers=('Driver', 'nunique'),
        laps=('LapNumber', 'count'),
    )
    .reset_index()
    .sort_values('RoundNumber')
)

print(f'Total rounds in file : {coverage["RoundNumber"].nunique()} / 24')
print(f'Total laps           : {final.shape[0]:,}')
print(f'Total drivers seen   : {final["Driver"].nunique()}')
print()
coverage

In [ ]:
print('Sample of loaded data:')
print(final.dtypes)
final.head(3)

---
## ✅ Done

The raw lap data is saved. Proceed to **[02_eda_lap_performance.ipynb](02_eda_lap_performance.ipynb)**.